# MRT Network Spectral Analysis



## Setup & Configuration
Edit `DATA_DIR` (or set `STATIONFLOW_DATA_DIR`) to point at your local data folder.

In [ ]:

import os
import csv
import warnings
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix, issparse
from scipy.sparse.csgraph import dijkstra, laplacian
from scipy.sparse.linalg import eigsh
from scipy.optimize import curve_fit

warnings.filterwarnings("ignore", category=RuntimeWarning)

# --- Local data location (no Google Drive dependency) ---------------------
DATA_DIR = Path(os.environ.get("STATIONFLOW_DATA_DIR", "resources"))

STN_DATA_PATH = DATA_DIR / "202504" / "node_simplified.csv"
OD_DATA_PATH = DATA_DIR / "202504" / "origin_destination_train_202504.csv"
STN_COOR_PATH = DATA_DIR / "stn_coor_010326.csv"

STATION_CODES = {"NE", "EW", "NS", "CC", "DT", "TE", "BP", "SW", "SE", "PW", "PE", "CE", "CG"}
MAX_STATIONS_PER_LINE = 50

CAPACITY = 300     # pax per train — adjust to your scenario
FREQUENCY = 12     # trains per hour — adjust to your scenario

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

for p in (STN_DATA_PATH, OD_DATA_PATH, STN_COOR_PATH):
    if not p.exists():
        print(f"NOTE: expected data file not found at '{p}'. "
              f"Set STATIONFLOW_DATA_DIR or place your data under '{DATA_DIR}/'.")


## Core Utilities

In [ ]:

def check_symmetry(matrix, label: str = "matrix") -> bool:
    dense = matrix.toarray() if issparse(matrix) else np.asarray(matrix)
    res = np.allclose(dense, dense.T)
    print(f"[{label}] symmetry check {'PASSED' if res else 'FAILED'}")
    return res


class Coordinate:
    __slots__ = ("lat", "lon")

    def __init__(self, lat: float, lon: float):
        self.lat = lat
        self.lon = lon


def haversine_distance(c1: Coordinate, c2: Coordinate) -> float:
    '''Great-circle distance in km between two Coordinates.'''
    R = 6378.137
    lat1, lon1, lat2, lon2 = map(np.radians, (c1.lat, c1.lon, c2.lat, c2.lon))
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return R * 2 * np.arcsin(np.sqrt(a))


## Station Index & Adjacency Matrix

**Complexity fix:** the original `search()` helper scanned every station
name (`O(N)`) for each of the ~650 `(line_code, stop_number)` combinations
tried while building the graph — `O(650·N)` overall. `_build_stop_code_lookup`
below builds a `{"NS1": "NS1", "EW3": "NS5/EW3", ...}` reverse-index once in
`O(N)`, after which every lookup is `O(1)`, making adjacency construction
`O(650)` regardless of network size.

In [ ]:

def build_station_index(stn_data_path: Path):
    with open(stn_data_path, newline="") as f:
        node_set = {row["PT_CODE"] for row in csv.DictReader(f)}
    nodes = sorted(node_set)
    node2index = {name: i for i, name in enumerate(nodes)}
    index2node = {i: name for i, name in enumerate(nodes)}
    print(f"Loaded {len(nodes)} unique stations.")
    return nodes, node2index, index2node


def _build_stop_code_lookup(node2index: dict) -> dict:
    '''Maps every individual stop code (e.g. 'EW3') to its full node name
    (e.g. 'NS5/EW3' for an interchange), built once in O(N).'''
    lookup = {}
    for name in node2index:
        for part in name.split("/"):
            lookup[part] = name
    return lookup


# Irregular / branch / loop connections not captured by sequential numbering.
DEFAULT_IRREGULAR_PAIRS = [
    ("CE1", "CC4"), ("NE17", "PE1"), ("NE17", "PE7"), ("NE17", "PW1"), ("NE17", "PW7"),
    ("NE16", "SW1"), ("NE16", "SW8"), ("NE16", "SE1"), ("NE16", "SE5"),
    ("BP13", "BP6"), ("EW4", "DT35"),
]


def build_unweighted_adjacency(nodes, node2index, irregular_pairs=DEFAULT_IRREGULAR_PAIRS):
    n = len(nodes)
    stop_lookup = _build_stop_code_lookup(node2index)
    adj = np.zeros((n, n), dtype=np.float64)

    for code in STATION_CODES:
        prev_node = None
        for i in range(1, MAX_STATIONS_PER_LINE + 1):
            node = stop_lookup.get(f"{code}{i}")
            if node is None:
                continue
            if prev_node is not None and prev_node != node:
                a, b = node2index[prev_node], node2index[node]
                adj[a, b] = adj[b, a] = 1
            prev_node = node

    for code_a, code_b in (irregular_pairs or []):
        a = node2index.get(stop_lookup.get(code_a, ""))
        b = node2index.get(stop_lookup.get(code_b, ""))
        if a is None or b is None:
            print(f"Warning: could not resolve irregular pair ({code_a}, {code_b}); skipped.")
            continue
        adj[a, b] = adj[b, a] = 1

    check_symmetry(adj, "unweighted_adj_matrix")
    print(f"Built adjacency matrix: {n} stations, {int(adj.sum() // 2)} edges.")
    return adj


In [ ]:

nodes, node2index, index2node = build_station_index(STN_DATA_PATH)
unweighted_adj_matrix = build_unweighted_adjacency(nodes, node2index)


## Station Coordinates

In [ ]:

def build_station_coordinates(stn_coor_path: Path, node2index: dict):
    raw = {}
    with open(stn_coor_path, newline="") as f:
        for row in csv.DictReader(f):
            raw[row["station_code"]] = Coordinate(float(row["lat"]), float(row["lon"]))

    stn_code_to_coor, missing = {}, []
    for full_name in node2index:
        if full_name in raw:
            stn_code_to_coor[full_name] = raw[full_name]
            continue
        for part in full_name.split("/"):
            if part in raw:
                stn_code_to_coor[full_name] = raw[part]
                break
        else:
            missing.append(full_name)

    if missing:
        preview = missing[:5]
        print(f"Warning: no coordinates for {len(missing)} station(s): {preview}{'...' if len(missing) > 5 else ''}")
    print(f"Aligned {len(stn_code_to_coor)}/{len(node2index)} station coordinates.")
    return stn_code_to_coor


stn_code_to_coor = build_station_coordinates(STN_COOR_PATH, node2index)


## OD Data & Weighted Graph

**De-duplication:** the original notebook rebuilt "route every OD pair over
the shortest path and accumulate trip counts onto edges" from scratch in
7 different cells (baseline weighting, edge/node/line/random/weighted
disruption, and network expansion). It's now the single `route_and_weight`
function below, used everywhere.

In [ ]:

def load_od_data(od_data_path: Path) -> np.ndarray:
    df = pd.read_csv(od_data_path, usecols=["ORIGIN_PT_CODE", "DESTINATION_PT_CODE", "TOTAL_TRIPS"])
    grouped = df.groupby(["ORIGIN_PT_CODE", "DESTINATION_PT_CODE"])["TOTAL_TRIPS"].sum().reset_index()
    return grouped.to_numpy()


def route_and_weight(base_unweighted_graph: np.ndarray, od_data_arr: np.ndarray,
                      node2index: dict, verbose: bool = False) -> np.ndarray:
    '''Route every OD pair over the shortest path of `base_unweighted_graph`
    (one all-pairs Dijkstra call) and accumulate trip counts onto edges.'''
    weighted = base_unweighted_graph.copy().astype(float)
    dist_matrix, pred_matrix = dijkstra(csgraph=base_unweighted_graph, directed=False,
                                         return_predecessors=True)

    total = len(od_data_arr)
    for count, (ori_code, des_code, trips) in enumerate(od_data_arr, start=1):
        u, v = node2index.get(ori_code), node2index.get(des_code)
        if u is None or v is None or u == v:
            continue
        if pred_matrix[u, v] == -9999:
            continue  # disconnected under the current (possibly disrupted) graph
        curr = v
        while curr != u:
            prev = pred_matrix[u, curr]
            if prev == -9999:
                break
            weighted[prev, curr] += trips
            weighted[curr, prev] += trips
            curr = prev
        if verbose and count % 5000 == 0:
            print(f"  routed {count}/{total} OD pairs")
    return weighted


od_data_arr = load_od_data(OD_DATA_PATH)
weighted_adj_matrix = route_and_weight(unweighted_adj_matrix, od_data_arr, node2index, verbose=True)
check_symmetry(weighted_adj_matrix, "weighted_adj_matrix")


## Spectral (Fiedler) Analysis

**Complexity fix:** `numpy.linalg.eigh` on the dense Laplacian computes the
*entire* eigenspectrum (`O(n^3)`) even though only the smallest 2–3
eigenvalues are ever used. `fiedler_pair` instead uses
`scipy.sparse.linalg.eigsh` with shift-invert (`sigma=0`), which converges to
just the requested eigenpairs in roughly `O(n^2)`–`O(n·nnz)`. This matters
most later, where the same computation is repeated per Monte-Carlo trial and
per network-expansion candidate.

In [ ]:

def fiedler_pair(weighted_graph, k: int = 3, use_sparse: bool = True):
    '''Return (fiedler_value, fiedler_vector, all_vals[:k], all_vecs[:, :k]).'''
    n = weighted_graph.shape[0]
    sparse_graph = weighted_graph if issparse(weighted_graph) else csr_matrix(weighted_graph)
    lap = laplacian(sparse_graph, normed=True, symmetrized=True)

    k_eff = max(min(k, n - 2), 2) if n > 10 else n
    if use_sparse and n > 10:
        try:
            vals, vecs = eigsh(lap, k=k_eff, sigma=0, which="LM")
            order = np.argsort(vals)
            vals, vecs = vals[order], vecs[:, order]
        except Exception:
            vals, vecs = np.linalg.eigh(lap.toarray())
    else:
        vals, vecs = np.linalg.eigh(lap.toarray())

    idx = 0
    while idx < len(vals) and np.isclose(vals[idx], 0.0, atol=1e-8):
        idx += 1
    idx = min(idx, len(vals) - 1)
    return vals[idx], vecs[:, idx], vals, vecs


fiedler_val, fiedler_vec, eigen_vals, eigen_vecs = fiedler_pair(weighted_adj_matrix)
print(f"Smallest positive (Fiedler) eigenvalue: {fiedler_val:.6f}")


### Visualization — Five Most Vulnerable Stations

In [ ]:

sec_eigen_vec_abs = np.abs(fiedler_vec)
top5_idx = np.argpartition(sec_eigen_vec_abs, -min(5, len(sec_eigen_vec_abs)))[-5:]
top5_names = [index2node[i] for i in top5_idx]

plt.figure(figsize=(10, 6))
plt.bar(top5_names, sec_eigen_vec_abs[top5_idx])
plt.title("Fiedler Eigenvector — Five Most Vulnerable Stations")
plt.xlabel("Station")
plt.ylabel("|Eigenvector value|")
plt.tight_layout()
plt.show()


## Utilization Factor (Queueing Theory)

In [ ]:

def utilization_factor(weighted_graph, capacity: float, frequency: float) -> np.ndarray:
    dense = weighted_graph.toarray() if issparse(weighted_graph) else np.asarray(weighted_graph)
    return dense.sum(axis=1) / (capacity * frequency)


baseline_util = utilization_factor(weighted_adj_matrix, CAPACITY, FREQUENCY)
print("Utilization factor (first 5 stations):", baseline_util[:5])


In [ ]:

normalized_fiedler_vec = sec_eigen_vec_abs / (np.linalg.norm(sec_eigen_vec_abs) or 1.0)
top10_idx = np.argsort(normalized_fiedler_vec)[-10:][::-1]

print("Top 10 stations by normalized |Fiedler eigenvector|:")
print("-" * 68)
for idx in top10_idx:
    print(f"  {index2node[idx]:<15} | {normalized_fiedler_vec[idx]:.6f}")


### Selected-Station Eigenvector Export

In [ ]:

TARGET_STATIONS = ["NE12/CC13", "NS9/TE2", "EW2/DT32", "EW8/CC9", "EW24/NS1"]

rows = []
for name in TARGET_STATIONS:
    if name in node2index:
        rows.append({"stn_name": name, "abs_eigen_vector": abs(fiedler_vec[node2index[name]])})
    else:
        print(f"Warning: station '{name}' not found; skipped.")

df_target = pd.DataFrame(rows)
df_target.to_csv(OUTPUT_DIR / "selected_stations_eigenvector_values.csv", index=False)
print(df_target)


## Logistic Regression — Hourly Station Throughput

Fits a two-peak (AM/PM) logistic curve to each target station's hourly
throughput, where throughput = trips whose routed shortest path passes
through that station (origin, destination, or transfer).

In [ ]:

def reconstruct_path(pred_matrix, start_idx, end_idx):
    if start_idx == end_idx:
        return [start_idx]
    if pred_matrix[start_idx, end_idx] == -9999:
        return None
    path, current = [end_idx], end_idx
    while current != start_idx:
        current = pred_matrix[start_idx, current]
        if current == -9999:
            return None
        path.append(current)
    path.reverse()
    return path


def build_hourly_station_throughput(df_hourly_od, node2index, index2node, pred_matrix):
    records = []
    for hour in sorted(df_hourly_od["hour"].unique()):
        df_hour = df_hourly_od[df_hourly_od["hour"] == hour]
        totals = {s: 0 for s in index2node.values()}
        for _, row in df_hour.iterrows():
            o, d, trips = row["ORIGIN_PT_CODE"], row["DESTINATION_PT_CODE"], row["TOTAL_TRIPS"]
            if o not in node2index or d not in node2index:
                continue
            path = reconstruct_path(pred_matrix, node2index[o], node2index[d])
            if path is None:
                continue
            for idx in path:
                totals[index2node[idx]] += trips
        for station, total in totals.items():
            records.append({"hour": hour, "station": station, "TOTAL_THROUGHPUT": total})
    return pd.DataFrame(records)


def two_peak_logistic(t, B, K1, gamma1, t1_start, t1_end, K2, gamma2, t2_start, t2_end):
    def sigmoid(x):
        return 1 / (1 + np.exp(-x))
    morning = K1 * np.minimum(sigmoid(gamma1 * (t - t1_start)), sigmoid(gamma1 * (t1_end - t)))
    evening = K2 * np.minimum(sigmoid(gamma2 * (t - t2_start)), sigmoid(gamma2 * (t2_end - t)))
    return B + morning + evening


np.random.seed(42)
df_od_full = pd.read_csv(OD_DATA_PATH, usecols=["ORIGIN_PT_CODE", "DESTINATION_PT_CODE", "TOTAL_TRIPS", "TIME_PER_HOUR"])
df_od_full = df_od_full.rename(columns={"TIME_PER_HOUR": "hour"})
df_od_hourly = df_od_full.groupby(["hour", "ORIGIN_PT_CODE", "DESTINATION_PT_CODE"])["TOTAL_TRIPS"].sum().reset_index()
df_od_hourly["hour"] = df_od_hourly["hour"].astype(float)

_, pred_matrix_baseline = dijkstra(csgraph=unweighted_adj_matrix, directed=False, return_predecessors=True)
df_station_hourly = build_hourly_station_throughput(df_od_hourly, node2index, index2node, pred_matrix_baseline)

plt.figure(figsize=(12, 6))
colors = ["#1f77b4", "#ebbda7", "#5B8E7D", "#9b9ab3", "#6D6875"]
for i, station in enumerate(TARGET_STATIONS):
    data = df_station_hourly[df_station_hourly["station"] == station].sort_values("hour").reset_index(drop=True)
    if data.empty or data["TOTAL_THROUGHPUT"].max() == 0:
        print(f"No routed throughput data for {station}; skipping curve fit.")
        continue

    t_data, y_data = data["hour"].values, data["TOTAL_THROUGHPUT"].values
    p0 = [float(y_data.min()), float(y_data.max() / 2), 1, 6, 10, float(y_data.max() / 2), 1, 17, 20]
    try:
        popt, _ = curve_fit(two_peak_logistic, t_data, y_data, p0=p0, maxfev=20000)
    except RuntimeError:
        popt = p0

    t_fit = np.linspace(0, 23, 400)
    y_fit = two_peak_logistic(t_fit, *popt)
    plt.plot(t_fit, y_fit, color=colors[i], linewidth=2.2, label=station)
    plt.scatter(t_data, y_data, color=colors[i], s=35, alpha=0.6, edgecolor="black", linewidth=0.5, zorder=5)

plt.axvspan(6, 9, color="red", alpha=0.1, label="Morning peak")
plt.axvspan(16, 19, color="green", alpha=0.1, label="Evening peak")
plt.title("Time-Based Logistic Regression of Station Throughput")
plt.xlabel("Time of Day")
plt.ylabel("Hourly Station Throughput")
plt.legend(title="Station", fontsize=9)
plt.tight_layout()
plt.show()


## Disruption Simulation Framework

**De-duplication:** the original notebook copy-pasted the same ~40-line
"reroute flow → Laplacian → eigendecompose → utilization delta → save CSV"
block for edge disruption, node disruption, line disruption, random
disruption, and weighted Monte-Carlo disruption. It's now the single
`analyze_disruption` function, reused by every scenario below.

**No blocking input:** the original cells called `input()` and would hang
(or error with `EOFError`) in any non-interactive run (e.g. `nbconvert
--execute`, CI, or "Run All"). Each scenario below is now driven by a plain
variable set at the top of its cell — edit it and re-run.

In [ ]:

def disrupt_edges(base_unweighted, node2index, edges):
    g = base_unweighted.copy()
    applied = []
    for a_name, b_name in edges:
        a, b = node2index.get(a_name), node2index.get(b_name)
        if a is None or b is None:
            print(f"Warning: unknown station in edge ({a_name}, {b_name}); skipped.")
            continue
        if g[a, b] == 0:
            print(f"Warning: no existing edge ({a_name}, {b_name}); skipped.")
            continue
        g[a, b] = g[b, a] = 0
        applied.append(f"{a_name}-{b_name}")
    return g, applied


def disrupt_stations(base_unweighted, node2index, stations):
    g = base_unweighted.copy()
    applied = []
    for name in stations:
        idx = node2index.get(name)
        if idx is None:
            print(f"Warning: unknown station '{name}'; skipped.")
            continue
        g[idx, :] = 0
        g[:, idx] = 0
        applied.append(name)
    return g, applied


def disrupt_line(base_unweighted, node2index, line_code):
    stations = [name for name in node2index if line_code in name]
    return disrupt_stations(base_unweighted, node2index, stations)


def disrupt_random_edges(base_unweighted, num_edges, rng):
    '''O(E) once to list edges, then O(num_edges) to sample -- no
    reject-and-retry random search loop like the original.'''
    g = base_unweighted.copy()
    edge_idx = np.argwhere(np.triu(base_unweighted) == 1)
    if num_edges > len(edge_idx):
        raise ValueError(f"Requested {num_edges} edges but only {len(edge_idx)} exist.")
    chosen = rng.choice(len(edge_idx), size=num_edges, replace=False)
    applied = []
    for i in chosen:
        a, b = edge_idx[i]
        g[a, b] = g[b, a] = 0
        applied.append((int(a), int(b)))
    return g, applied


def analyze_disruption(disrupted_unweighted, od_data_arr, node2index, index2node,
                        baseline_fiedler_val, baseline_fiedler_vec, baseline_util,
                        capacity=CAPACITY, frequency=FREQUENCY):
    '''Shared post-disruption metrics used by every scenario below.'''
    weighted = route_and_weight(disrupted_unweighted, od_data_arr, node2index)
    fval, fvec, _, _ = fiedler_pair(weighted)
    util = utilization_factor(weighted, capacity, frequency)

    delta_evec = np.abs(fvec) - np.abs(baseline_fiedler_vec)
    delta_util = util - baseline_util

    return {
        "weighted_graph": weighted,
        "fiedler_value": fval,
        "fiedler_vector": fvec,
        "utilization": util,
        "avg_abs_change_eigenvector": float(np.mean(np.abs(delta_evec))),
        "delta_utilization": delta_util,
        "avg_change_utilization": float(np.mean(delta_util[delta_util != 0])) if np.any(delta_util != 0) else 0.0,
    }


def save_disruption_result(label: str, applied, result: dict, filename: str):
    df_out = pd.DataFrame({
        "Station/Edge Disrupted": [", ".join(map(str, applied))] * len(nodes),
        "Station": [index2node[i] for i in range(len(nodes))],
        "Pre-Disruption Fiedler Value": [fiedler_val] * len(nodes),
        "Post-Disruption Fiedler Value": [result["fiedler_value"]] * len(nodes),
        "Avg |Eigenvector| Change": [result["avg_abs_change_eigenvector"]] * len(nodes),
        "Utilization Factor Change": result["delta_utilization"],
    })
    path = OUTPUT_DIR / filename
    df_out.to_csv(path, index=False)
    print(f"[{label}] Fiedler value {fiedler_val:.4f} -> {result['fiedler_value']:.4f}  "
          f"(saved to {path})")


### Scenario 1 — Specific Edge Disruption

In [ ]:

# Edit this list to change which edges are disrupted, e.g.:
# EDGES_TO_DISRUPT = [("EW24/NS1", "NS2"), ("NE1", "CC2")]
EDGES_TO_DISRUPT = [("NS1", "NS2")]

disrupted_graph, applied_edges = disrupt_edges(unweighted_adj_matrix, node2index, EDGES_TO_DISRUPT)
if applied_edges:
    result = analyze_disruption(disrupted_graph, od_data_arr, node2index, index2node,
                                 fiedler_val, fiedler_vec, baseline_util)
    save_disruption_result("Edge disruption", applied_edges, result, "edge_disruption_analysis.csv")
else:
    print("No edges were disrupted; nothing to analyze.")


### Scenario 2 — Station (Node) Disruption

In [ ]:

# Edit this list to change which stations are disrupted, e.g.:
# STATIONS_TO_DISRUPT = ["NE12/CC13", "NS9/TE2"]
STATIONS_TO_DISRUPT = [nodes[0]]

disrupted_graph, applied_stations = disrupt_stations(unweighted_adj_matrix, node2index, STATIONS_TO_DISRUPT)
if applied_stations:
    result = analyze_disruption(disrupted_graph, od_data_arr, node2index, index2node,
                                 fiedler_val, fiedler_vec, baseline_util)
    save_disruption_result("Station disruption", applied_stations, result, "station_disruption_analysis.csv")


### Scenario 3 — Line Disruption

In [ ]:

# Edit to change which line is fully disrupted (matches any station whose
# name contains this code, e.g. "NS" for the North-South line).
LINE_TO_DISRUPT = "NS"

disrupted_graph, applied_stations = disrupt_line(unweighted_adj_matrix, node2index, LINE_TO_DISRUPT)
if applied_stations:
    result = analyze_disruption(disrupted_graph, od_data_arr, node2index, index2node,
                                 fiedler_val, fiedler_vec, baseline_util)
    save_disruption_result(f"Line '{LINE_TO_DISRUPT}' disruption", applied_stations, result,
                            f"{LINE_TO_DISRUPT}_line_disruption_analysis.csv")


### Scenario 4 — Basic Random Disruption

In [ ]:

NUM_RANDOM_EDGES = 2
rng = np.random.default_rng(42)

disrupted_graph, applied_edges_idx = disrupt_random_edges(unweighted_adj_matrix, NUM_RANDOM_EDGES, rng)
applied_names = [f"{index2node[a]}-{index2node[b]}" for a, b in applied_edges_idx]
print("Randomly disrupted edges:", applied_names)

result = analyze_disruption(disrupted_graph, od_data_arr, node2index, index2node,
                             fiedler_val, fiedler_vec, baseline_util)
save_disruption_result("Random disruption", applied_names, result, "random_disruption_analysis.csv")

for i in range(len(nodes)):
    change = result["delta_utilization"][i]
    if change != 0:
        print(f"  {index2node[i]}: utilization change {change:+.4f}")


### Scenario 5 — Weighted (Importance-Sampled) Monte-Carlo Disruption

Stations are sampled for disruption proportionally to their baseline Fiedler
eigenvector magnitude (i.e. more "important" stations are disrupted more
often), and we track the average network impact per station.

**Complexity:** `NUM_TRIALS` full disruption analyses are run, each
`O(V log V + E)` for routing plus `O(n^2)`–`O(n·nnz)` for the sparse
eigendecomposition (vs. `O(n^3)` in the original dense version). The
original notebook used 50,000 trials and noted this could take up to 20
hours; with the sparse eigensolver a few thousand trials is enough for
stable averages and finishes in well under a minute for a network of a few
hundred stations. Raise `NUM_TRIALS` if you have time to spare.

In [ ]:

NUM_TRIALS = 500          # was 50,000 in the original — see note above
MAX_EDGES_PER_TRIAL = 3   # was "up to 5" in the original
RNG_SEED = 7

rng = np.random.default_rng(RNG_SEED)

# Sampling weights: proportional to |Fiedler eigenvector| per station, i.e.
# more central/vulnerable stations are disrupted more often.
station_weights = np.abs(fiedler_vec)
station_weights = station_weights / station_weights.sum() if station_weights.sum() > 0 else np.full(len(nodes), 1 / len(nodes))

edge_list = np.argwhere(np.triu(unweighted_adj_matrix) == 1)  # (E, 2), built once

trial_fiedler_vals = np.empty(NUM_TRIALS)
trial_util = np.empty((NUM_TRIALS, len(nodes)))
trial_disrupted_stations = []

for t in range(NUM_TRIALS):
    n_edges = rng.integers(1, MAX_EDGES_PER_TRIAL + 1)
    # Prefer edges touching higher-weight stations, without a reject/retry loop:
    endpoint_weight = station_weights[edge_list[:, 0]] + station_weights[edge_list[:, 1]]
    edge_probs = endpoint_weight / endpoint_weight.sum()
    chosen = rng.choice(len(edge_list), size=min(n_edges, len(edge_list)), replace=False, p=edge_probs)

    g = unweighted_adj_matrix.copy()
    disrupted_stns = set()
    for i in chosen:
        a, b = edge_list[i]
        g[a, b] = g[b, a] = 0
        disrupted_stns.add(int(a))
        disrupted_stns.add(int(b))
    trial_disrupted_stations.append(disrupted_stns)

    weighted = route_and_weight(g, od_data_arr, node2index)
    fval, _, _, _ = fiedler_pair(weighted)
    trial_fiedler_vals[t] = fval
    trial_util[t] = utilization_factor(weighted, CAPACITY, FREQUENCY)

    if (t + 1) % 100 == 0:
        print(f"  completed {t + 1}/{NUM_TRIALS} trials")

print(f"\nAverage Fiedler value across {NUM_TRIALS} trials: {trial_fiedler_vals.mean():.6f} "
      f"(baseline: {fiedler_val:.6f})")

# Per-station average impact when that station was involved in a disruption
n = len(nodes)
eig_sum, eig_cnt = np.zeros(n), np.zeros(n)
util_sum, util_cnt = np.zeros(n), np.zeros(n)
mean_util = trial_util.mean(axis=0)

for t, stns in enumerate(trial_disrupted_stations):
    for s in stns:
        eig_sum[s] += trial_fiedler_vals[t]
        eig_cnt[s] += 1
        util_sum[s] += trial_util[t, s] - baseline_util[s]
        util_cnt[s] += 1

rows = []
for i in range(n):
    if eig_cnt[i] == 0:
        continue
    rows.append({
        "Station": index2node[i],
        "Pre-Disruption Fiedler Value": fiedler_val,
        "Avg Post-Disruption Fiedler Value": eig_sum[i] / eig_cnt[i],
        "Avg Own Utilization Change": util_sum[i] / util_cnt[i],
        "Times Disrupted": int(eig_cnt[i]),
    })

df_mc = pd.DataFrame(rows).sort_values("Avg Post-Disruption Fiedler Value")
df_mc.to_csv(OUTPUT_DIR / "monte_carlo_disruption_summary.csv", index=False)
print(df_mc.head(10).to_string(index=False))


## Network Expansion Analysis

Scores every *candidate new edge* (any currently-unconnected station pair
closer than the median existing edge length) by how much it would improve
network connectivity (raise the Fiedler value) and reduce peak utilization.

**Complexity:** the number of candidate pairs is `O(n^2)` and each candidate
requires a full re-routing + spectral analysis, so the total cost is
`O(n^2 · (E + n^2))` in the worst case — this is inherent to exhaustively
scoring every candidate edge, not something that can be removed without
changing the question being asked. Two practical mitigations are applied:
distances are vectorized with NumPy instead of a nested Python loop, and
`MAX_CANDIDATES` caps the search (randomly sampled) to keep runtime bounded
on large networks — set it to `None` to score every candidate exhaustively.

In [ ]:

MAX_CANDIDATES = 200  # cap on candidate edges scored; set to None to score all

def candidate_edges_by_distance(nodes, node2index, index2node, unweighted_adj_matrix, stn_code_to_coor):
    n = len(nodes)
    coords = [stn_code_to_coor.get(index2node[i]) for i in range(n)]
    have_coord = np.array([c is not None for c in coords])

    lat = np.array([c.lat if c else 0.0 for c in coords])
    lon = np.array([c.lon if c else 0.0 for c in coords])

    # Vectorized haversine distance matrix (replaces the original nested
    # Python double-loop distance calculation).
    R = 6378.137
    lat_r, lon_r = np.radians(lat), np.radians(lon)
    dlat = lat_r[:, None] - lat_r[None, :]
    dlon = lon_r[:, None] - lon_r[None, :]
    a = np.sin(dlat / 2) ** 2 + np.cos(lat_r[:, None]) * np.cos(lat_r[None, :]) * np.sin(dlon / 2) ** 2
    dist_matrix = R * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

    iu = np.triu_indices(n, k=1)
    valid = have_coord[iu[0]] & have_coord[iu[1]]

    existing_mask = unweighted_adj_matrix[iu] == 1
    existing_lengths = dist_matrix[iu][existing_mask & valid]
    median_length = np.median(existing_lengths) if len(existing_lengths) else np.inf

    candidate_mask = (~existing_mask) & valid & (dist_matrix[iu] <= median_length)
    candidates = list(zip(iu[0][candidate_mask], iu[1][candidate_mask]))
    print(f"Median existing edge length: {median_length:.2f} km")
    print(f"Candidate edges within that distance: {len(candidates)}")
    return candidates


def analyze_network_expansion(candidates, unweighted_adj_matrix, od_data_arr, node2index, index2node,
                               baseline_fiedler_val, baseline_fiedler_vec, baseline_util,
                               capacity=CAPACITY, frequency=FREQUENCY):
    results = []
    for k, (i, j) in enumerate(candidates, start=1):
        trial_graph = unweighted_adj_matrix.copy()
        trial_graph[i, j] = trial_graph[j, i] = 1

        weighted = route_and_weight(trial_graph, od_data_arr, node2index)
        fval, fvec, _, _ = fiedler_pair(weighted)
        if np.dot(fvec, baseline_fiedler_vec) < 0:
            fvec = -fvec
        util = utilization_factor(weighted, capacity, frequency)

        avg_change_evec = float(np.mean(np.abs(fvec - baseline_fiedler_vec)))
        delta_util = util - baseline_util
        decreased = delta_util[delta_util < 0]
        avg_change_util_dec = float(decreased.mean()) if len(decreased) else 0.0

        results.append({
            "EDGE_BETWEEN_STN": f"{index2node[i]} - {index2node[j]}",
            "original_eigen_value": baseline_fiedler_val,
            "current_eigen_value": fval,
            "avg_change_eigen_vector": avg_change_evec,
            "avg_change_util_factor_decreased": avg_change_util_dec,
        })
        if k % 50 == 0:
            print(f"  scored {k}/{len(candidates)} candidate edges")

    return pd.DataFrame(results)


candidates = candidate_edges_by_distance(nodes, node2index, index2node, unweighted_adj_matrix, stn_code_to_coor)
if MAX_CANDIDATES is not None and len(candidates) > MAX_CANDIDATES:
    rng = np.random.default_rng(0)
    sel = rng.choice(len(candidates), size=MAX_CANDIDATES, replace=False)
    candidates = [candidates[i] for i in sel]
    print(f"Sampled down to {MAX_CANDIDATES} candidates for tractability.")

df_expansion = analyze_network_expansion(candidates, unweighted_adj_matrix, od_data_arr, node2index, index2node,
                                          fiedler_val, fiedler_vec, baseline_util)
df_expansion.to_csv(OUTPUT_DIR / "network_expansion_analysis.csv", index=False)
print(f"Analysis complete. {len(df_expansion)} candidate edges saved to network_expansion_analysis.csv")


### Rank Expansion Candidates by Composite Score

In [ ]:

def rank_by_composite_score(df, w_eigval=1/3, w_eigvec=1/3, w_util=1/3, top_k=10):
    df = df.copy()
    df["score_eigval"] = df["current_eigen_value"].rank(ascending=False, method="average")
    df["score_eigval"] = 1 - (df["score_eigval"] - 1) / max(len(df) - 1, 1)

    df["score_eigvec"] = df["avg_change_eigen_vector"].rank(ascending=True, method="average")
    df["score_eigvec"] = 1 - (df["score_eigvec"] - 1) / max(len(df) - 1, 1)

    df["score_util"] = df["avg_change_util_factor_decreased"].rank(ascending=True, method="average")
    df["score_util"] = 1 - (df["score_util"] - 1) / max(len(df) - 1, 1)

    df["composite_score"] = w_eigval * df["score_eigval"] + w_eigvec * df["score_eigvec"] + w_util * df["score_util"]
    return df.sort_values("composite_score", ascending=False).head(top_k)


if len(df_expansion):
    top_candidates = rank_by_composite_score(df_expansion)
    cols = ["EDGE_BETWEEN_STN", "current_eigen_value", "avg_change_eigen_vector",
            "avg_change_util_factor_decreased", "composite_score"]
    pd.set_option("display.max_colwidth", 60)
    print(top_candidates[cols].to_string(index=False))
else:
    print("No candidate edges to rank.")


## Diagram Rendering

In [ ]:

def plot_network(unweighted_adj_matrix, node2index, index2node, stn_code_to_coor,
                  highlight_stations=None, highlight_edges=None, title="MRT Network Graph"):
    highlight_stations = set(highlight_stations or [])
    plt.figure(figsize=(16, 12))

    for stn_code, idx in node2index.items():
        coord = stn_code_to_coor.get(stn_code)
        if coord is None:
            continue
        if stn_code in highlight_stations:
            plt.plot(coord.lon, coord.lat, 'o', color='orangered', markersize=14,
                      markeredgecolor='black', markeredgewidth=2, zorder=5)
            plt.text(coord.lon, coord.lat, stn_code, fontsize=11, ha='right', va='bottom',
                      color='darkblue', weight='bold', zorder=6)
        else:
            plt.plot(coord.lon, coord.lat, 'o', color='steelblue', markersize=6, zorder=1)

    n = len(index2node)
    for i in range(n):
        for j in range(i + 1, n):
            if unweighted_adj_matrix[i, j] != 1:
                continue
            stn_i, stn_j = index2node[i], index2node[j]
            c_i, c_j = stn_code_to_coor.get(stn_i), stn_code_to_coor.get(stn_j)
            if c_i is None or c_j is None:
                continue
            plt.plot([c_i.lon, c_j.lon], [c_i.lat, c_j.lat], 'k-', linewidth=1.5, alpha=0.4, zorder=0)

    for a, b in (highlight_edges or []):
        c_a, c_b = stn_code_to_coor.get(a), stn_code_to_coor.get(b)
        if c_a and c_b:
            plt.plot([c_a.lon, c_b.lon], [c_a.lat, c_b.lat], color='darkred', linewidth=3, zorder=3)

    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_network(unweighted_adj_matrix, node2index, index2node, stn_code_to_coor,
             title="Original MRT Network Graph")


In [ ]:

# Example: highlight the top expansion candidate from the ranking above, if any.
if len(df_expansion):
    best_edge = rank_by_composite_score(df_expansion, top_k=1).iloc[0]["EDGE_BETWEEN_STN"]
    a_name, b_name = [s.strip() for s in best_edge.split(" - ")]
    plot_network(unweighted_adj_matrix, node2index, index2node, stn_code_to_coor,
                 highlight_stations=[a_name, b_name], highlight_edges=[(a_name, b_name)],
                 title=f"Improved Network — Highlighted Candidate Edge: {best_edge}")


## Pareto Frontier — Capacity vs Frequency Trade-off

In [ ]:

NUM_SAMPLES = 100_000
COST_PER_NEW_EDGE = 520_000_000
NUM_NEW_EDGE = 2
OPERATIONAL_COST_PER_CAPACITY = 2.15
BETA, ALPHA = 0.7, 1
FREQ_MIN, FREQ_MAX = 1, 50

MATRIX_MODEL = weighted_adj_matrix.copy()
rng = np.random.default_rng(0)

raw_caps = rng.choice((320 * 6, 310 * 3), NUM_SAMPLES)
raw_freqs = rng.uniform(FREQ_MIN, FREQ_MAX, NUM_SAMPLES)


def check_constraints(c, f):
    restriction_a = COST_PER_NEW_EDGE * NUM_NEW_EDGE <= 1_487_559_500
    restriction_b = OPERATIONAL_COST_PER_CAPACITY * c * f * 24 * 365 <= 301_440_500
    restriction_c = f * c <= 301_440_500 / OPERATIONAL_COST_PER_CAPACITY / 24 / 365
    restriction_d = (f >= 1) & (f <= 50)
    return restriction_a & restriction_b & restriction_c & restriction_d


valid = check_constraints(raw_caps, raw_freqs)
caps, freqs = raw_caps[valid], raw_freqs[valid]
print(f"Generated {NUM_SAMPLES} pairs. Valid pairs after constraints: {len(caps)}")


def calculate_metrics(c_arr, f_arr, matrix_model):
    costs = COST_PER_NEW_EDGE * NUM_NEW_EDGE + OPERATIONAL_COST_PER_CAPACITY * (BETA * c_arr + f_arr * ALPHA)
    flow_per_node = matrix_model.sum(axis=1)  # shared across all (c, f) pairs -- computed once
    # Vectorized over all samples at once instead of a per-pair Python loop.
    satisfactions = flow_per_node.sum() / (c_arr * f_arr * len(flow_per_node))
    return costs, satisfactions


raw_cost, raw_sat = calculate_metrics(caps, freqs, MATRIX_MODEL)


def normalize(x):
    rng_ = x.max() - x.min()
    return np.zeros_like(x) if rng_ == 0 else (x - x.min()) / rng_


cost_norm, sat_norm = normalize(raw_cost), normalize(raw_sat)


def get_pareto_frontier(costs, satisfactions):
    '''Vectorized non-dominated-set computation (minimize both objectives).'''
    n = len(costs)
    mask = np.ones(n, dtype=bool)
    for i in range(n):
        if not mask[i]:
            continue
        dominators = (costs <= costs[i]) & (satisfactions <= satisfactions[i])
        dominators[i] = False
        if dominators.any():
            mask[i] = False
    return mask


pareto_mask = get_pareto_frontier(cost_norm, sat_norm)
p_cost, p_sat = cost_norm[pareto_mask], sat_norm[pareto_mask]
p_caps, p_freqs = caps[pareto_mask], freqs[pareto_mask]

diff = np.sqrt(p_sat ** 2 + p_cost ** 2)
best_local = np.argmin(diff)
best_global = np.where(pareto_mask)[0][best_local]

best_solution = {
    "capacity": caps[best_global], "frequency": freqs[best_global],
    "cost_norm": cost_norm[best_global], "sat_norm": sat_norm[best_global],
    "raw_cost": raw_cost[best_global], "raw_sat": raw_sat[best_global],
}

plt.figure(figsize=(10, 6))
plt.scatter(sat_norm, cost_norm, c='gray', s=5, alpha=0.5, label='Valid Solutions')
order = np.argsort(p_sat)
plt.plot(p_sat[order], p_cost[order], c='red', linewidth=2, label='Pareto Frontier')
plt.scatter(best_solution['sat_norm'], best_solution['cost_norm'], c='green', marker='*', s=300,
            edgecolors='black', zorder=10, label='Balanced Solution')
plt.title(f'Pareto Optimization: Satisfaction vs Cost (N={len(caps)})')
plt.xlabel('Satisfaction Factor (Normalized, smaller is better)')
plt.ylabel('Cost Factor (Normalized, smaller is better)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()

print("=" * 40)
print("OPTIMIZATION RESULTS")
print("=" * 40)
print(f"  Capacity  : {best_solution['capacity']:.0f} pax/train")
print(f"  Frequency : {best_solution['frequency']:.2f} trains/hour "
      f"(~{60/best_solution['frequency']:.1f} min headway)")
print(f"  Total Flow: {best_solution['capacity'] * best_solution['frequency']:.0f} pax/hour")
